In [1]:
%pip install python-dotenv openai datasets math_verify tqdm torch aiolimiter

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
import os
from openai import AsyncOpenAI
from aiolimiter import AsyncLimiter
from asyncio import Semaphore
from math_verify import parse
from dataclasses import dataclass

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
MODEL_ID = "openai/gpt-oss-20b"

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=0,
)

@dataclass
class MyCompletionChoice:
	reasoning_content: str = ""
	content: str = ""

limiter = AsyncLimiter(20)
semaphore = Semaphore(80)
async def create_completion(*args, **kwargs):
	kwargs["stream"] = True
	while True:
		try:
			async with limiter:
				async with semaphore:
					choices: list[MyCompletionChoice] = []
					async for chunk in await client.chat.completions.create(*args, **kwargs):
						for choice in chunk.choices:
							while len(choices) <= choice.index:
								choices.append(MyCompletionChoice())

							if delta := getattr(choice.delta, "reasoning_content", None):
								choices[choice.index].reasoning_content += delta

							if delta := getattr(choice.delta, "content", None):
								choices[choice.index].content += delta
					return choices
		except:
			pass

prompt = "What is 13 times 17? Box your answer."
gold = "221"

choices = await create_completion(
	model=MODEL_ID,
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(choices):
  parsed_answer = parse(choice.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Choice {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(f"<think>{choice.reasoning_content}</think>{choice.content}")

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Choice 1: 221 (correct) ********************
<think>We just need to give multiplication 13*17. 13*17 = 221. They want to box the answer. So we need to output like:

```
 ▓▓▓▓  
221
```

But they'd like "Box your answer." So we can show the answer in a box, e.g., using ascii.

We can also use markdown with a code block but that's not a box. We can mimic a box using maybe a matrix.

Alternatively, we can produce something like:

```
┌───────┐
│   221 │
└───────┘
```

Thus answer is 221. Probably that is enough.</think>```
┌───────┐
│   221 │
└───────┘
```

******************** Choice 2: 221 (correct) ********************
<think>We need to respond with the answer to 13 times 17, then "Box your answer." So we just need to produce the multiplication: 13*17=221. Then we need to box it. Likely we put it in something like a pair of brackets or show a box. We can use an ASCII box: e.g.

┌

In [ ]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train")
ds

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/open-r1/OpenR1-Math-220k/e4e141ec9dea9f8326f4d347be56105859b2bd68/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/open-r1/OpenR1-Math-220k/open-r1/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/revision/e4e141ec9dea9f8326f4d347be56105859b2bd68 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/.huggingface.yaml "HTTP/1.1 404 

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/tree/e4e141ec9dea9f8326f4d347be56105859b2bd68/data?recursive=true&expand=false "HTTP/1.1 200 OK"


DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [6]:
from tqdm.contrib.logging import logging_redirect_tqdm
from tqdm.asyncio import tqdm_asyncio
from math_verify import verify
from datasets import Dataset

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"results": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(create_completion(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	with logging_redirect_tqdm():
		completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
	for prompt, choices, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		results = []
		for choice in choices:
			answer = parse(choice.content)
			result = verify(gold, answer)

			outputs.append(f"<think>{choice.reasoning_content}</think>{choice.content}")
			results.append(result)

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["results"].append(results)
	return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model=MODEL_ID,
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10,
    },
)
example_ds[:]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
Creating completions: 100%|██████████| 2/2 [00:14<00:00,  7.23s/it]


{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>User: "Pick an random integer from 1 to 3. Don\'t pick 2. Box your answer."\n\nThey want a random integer from {1,3} presumably. Should output 1 or 3. Probably choose 1 or 3 randomly? Possibly they want random selection but can\'t pick 2. The answer should be boxed? So e.g.:\n\n```\n[box] 1 [/box]\n```\n\nWe can just pick randomly: let\'s choose 3. Or 1. The instruction: "Pick an random integer from 1 to 3. Don\'t pick 2." So we just give a random selection: 1 or 3. "Box your answer." So maybe a simple formatted box. Could use LaTeX: \\boxed{1}.\n\nThus respond with maybe \\boxed{3}. That\'s good.</think>\\[\n\\boxed{3}\n\\]',
   '<think>User requesting: "Pick a random integer from 1 to 3. Don\'t pick 2. Box your answer." They want a random integer between 1 and 3 not equal to 2. So possible choices 1 or 3. They want it boxed. We must comply 

In [7]:
input_ds = ds.shuffle().select(range(128))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model=MODEL_ID,
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15,
    },
)
output_ds

Creating completions:   0%|          | 0/128 [00:00<?, ?it/s]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/

CancelledError: 

In [ ]:
output_ds.to_parquet("reg_grpo_128.parquet")

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

100403699